# 3회차 실습: Genesis + Franka Starter for Manipulation

이번 시간부터는 수식 위주의 장난감 예제를 벗어나서, 실제 simulator 안에서 Franka를 움직여 봅니다.

다만 여기서 한 번에 너무 많은 것을 하려 하면 수업이 바로 무거워집니다.
그래서 이번 회차는 `starter code를 읽고, 주어진 target pose들을 안정적으로 따라가게 만드는 것`에 집중합니다.

오늘 실습의 핵심은 아래 세 가지입니다.

- Genesis scene을 만들고 Franka를 불러오기
- end-effector target pose를 joint command로 바꾸기
- 여러 pose를 부드러운 joint trajectory로 연결하고, 각 pose에서 1초씩 정지하기


## 이번 주 흐름 먼저 보기

이번 실습은 final project로 가기 위한 첫 simulator 주차입니다.

지금 당장 pick-and-place 전체를 한 번에 구현하는 것이 목표는 아닙니다.
먼저 아래 흐름이 안정적으로 돌아가는 것이 더 중요합니다.

1. scene 안에 table, cube, obstacle, Franka를 올린다.
2. 현재 joint state와 end-effector pose를 읽는다.
3. target pose를 하나 정한다.
4. 그 pose에 대응하는 joint target을 만든다.
5. joint trajectory를 부드럽게 만들어 실행한다.
6. 여러 pose를 순서대로 방문하고, 각 지점에서 잠시 정지한다.

이번 과제에서는 이 흐름만 깔끔하게 되면 충분합니다.
그 다음 주제인 pre-grasp, pick, place, obstacle avoidance는 이 위에 쌓으면 됩니다.


## 0. Imports and Utility Functions

In [ ]:
import numpy as np
import genesis as gs
from dataclasses import dataclass, field

np.set_printoptions(precision=4, suppress=True)


def to_numpy(x):
    if isinstance(x, np.ndarray):
        return x.astype(float)
    if hasattr(x, "detach"):
        x = x.detach()
    if hasattr(x, "cpu"):
        x = x.cpu()
    if hasattr(x, "numpy"):
        return x.numpy().astype(float)
    return np.asarray(x, dtype=float)


## 1. Starter Scene 만들기

이 셀들은 이번 시간의 starter code입니다.
학생들은 여기서 scene 생성, Franka 로드, marker 시각화, gripper open/close 같은 boilerplate를 처음부터 다 작성할 필요는 없습니다.

대신 아래 코드가 어떤 역할을 하는지 읽고, 그 위에서 `pose -> IK -> smooth trajectory` 부분을 직접 채우는 것이 핵심입니다.


In [ ]:
@dataclass
class Week3Config:
    dt: float = 0.01
    settle_steps: int = 120
    hold_time_sec: float = 1.0
    steps_per_segment: int = 120
    gripper_open: float = 0.04
    gripper_closed: float = 0.0
    home_qpos: tuple[float, ...] = (
        0.0,
        -0.65,
        0.0,
        -2.0,
        0.0,
        1.35,
        0.78,
        0.04,
        0.04,
    )
    table_pos: tuple[float, float, float] = (0.45, 0.0, 0.20)
    table_size: tuple[float, float, float] = (0.70, 1.10, 0.40)
    cube_pos: tuple[float, float, float] = (0.58, -0.18, 0.43)
    cube_size: float = 0.045
    obstacle_pos: tuple[float, float, float] = (0.52, 0.0, 0.50)
    obstacle_size: tuple[float, float, float] = (0.06, 0.26, 0.12)
    pose_markers: list[tuple[tuple[float, float, float], tuple[float, float, float]]] = field(
        default_factory=lambda: [
            ((0.52, -0.22, 0.58), (0.95, 0.30, 0.30)),
            ((0.62,  0.00, 0.48), (0.30, 0.80, 0.30)),
            ((0.44,  0.22, 0.60), (0.30, 0.40, 0.95)),
        ]
    )


cfg = Week3Config()


In [ ]:
def setup_franka(robot):
    robot.set_dofs_kp(np.array([4500, 4500, 3500, 3500, 2000, 2000, 2000, 100, 100]))
    robot.set_dofs_kv(np.array([450, 450, 350, 350, 200, 200, 200, 10, 10]))
    robot.set_dofs_force_range(
        np.array([-87, -87, -87, -87, -12, -12, -12, -100, -100]),
        np.array([87, 87, 87, 87, 12, 12, 12, 100, 100]),
    )


def build_scene(cfg, show_viewer=True, backend=gs.gpu):
    gs.init(backend=backend)

    scene = gs.Scene(
        viewer_options=gs.options.ViewerOptions(
            camera_pos=(2.4, -1.2, 1.5),
            camera_lookat=(0.45, 0.0, 0.45),
            camera_fov=35,
            max_FPS=60,
        ),
        sim_options=gs.options.SimOptions(dt=cfg.dt),
        rigid_options=gs.options.RigidOptions(
            box_box_detection=True,
            enable_collision=True,
            enable_joint_limit=True,
        ),
        show_viewer=show_viewer,
    )

    scene.add_entity(gs.morphs.Plane(), name="ground")

    table = scene.add_entity(
        gs.morphs.Box(pos=cfg.table_pos, size=cfg.table_size, fixed=True),
        surface=gs.surfaces.Plastic(color=(0.76, 0.73, 0.68)),
        name="table",
    )

    cube = scene.add_entity(
        gs.morphs.Box(pos=cfg.cube_pos, size=(cfg.cube_size, cfg.cube_size, cfg.cube_size)),
        material=gs.materials.Rigid(rho=250),
        surface=gs.surfaces.Plastic(color=(0.85, 0.28, 0.28)),
        name="cube",
    )

    obstacle = scene.add_entity(
        gs.morphs.Box(pos=cfg.obstacle_pos, size=cfg.obstacle_size, fixed=True),
        surface=gs.surfaces.Plastic(color=(0.20, 0.45, 0.92)),
        name="obstacle",
    )

    markers = []
    marker_size = (0.035, 0.035, 0.035)
    for idx, (pos, color) in enumerate(cfg.pose_markers):
        marker = scene.add_entity(
            gs.morphs.Box(pos=pos, size=marker_size, fixed=True, collision=False),
            surface=gs.surfaces.Plastic(color=color),
            name=f"pose_marker_{idx}",
        )
        markers.append(marker)

    robot = scene.add_entity(
        gs.morphs.MJCF(file="xml/franka_emika_panda/panda.xml"),
        name="franka",
    )

    scene.build()
    setup_franka(robot)
    robot.set_qpos(np.array(cfg.home_qpos))
    for _ in range(cfg.settle_steps):
        scene.step()

    entities = {
        "table": table,
        "cube": cube,
        "obstacle": obstacle,
        "markers": markers,
        "robot": robot,
    }
    return scene, entities


In [ ]:
scene, entities = build_scene(cfg, show_viewer=True)
robot = entities["robot"]
ee_link = robot.get_link("hand")
print("Scene ready.")
print("Markers:", [m.name for m in entities["markers"]])


## 2. 현재 robot state 읽기

실제 simulator 실습에서는 항상 `지금 로봇이 어디에 있는지`를 읽을 수 있어야 합니다.

아래 셀에서 먼저 확인할 것은 두 가지입니다.

- 현재 joint state `q`
- 현재 end-effector pose `(position, quaternion)`

이 두 정보는 나중에 trajectory 시작점이나 IK 초기값을 정할 때 계속 사용하게 됩니다.


In [ ]:
def get_arm_and_finger_slices(robot):
    q_dim = len(to_numpy(robot.get_qpos()))
    arm = slice(0, q_dim - 2)
    fingers = slice(q_dim - 2, q_dim)
    return arm, fingers


def read_robot_state(robot, ee_link):
    q = to_numpy(robot.get_qpos())
    ee_pos = to_numpy(ee_link.get_pos())
    ee_quat = to_numpy(ee_link.get_quat())
    return q, ee_pos, ee_quat


q_now, ee_pos_now, ee_quat_now = read_robot_state(robot, ee_link)
print("Current q =", q_now)
print("Current ee position =", ee_pos_now)
print("Current ee quaternion =", ee_quat_now)


## 3. Starter Utility: joint command와 gripper command

이 부분도 starter code입니다.

이번 실습에서는 학생들이 low-level control gain을 건드리기보다,
`joint target을 만들고 control_dofs_position으로 보내는 흐름`에 익숙해지는 것이 더 중요합니다.

아래 함수들은 다음 역할을 합니다.

- gripper open / close
- target joint를 잠시 유지하기
- joint trajectory array를 순서대로 실행하기


In [ ]:
def hold_current_target(scene, robot, q_target, steps):
    for _ in range(steps):
        robot.control_dofs_position(q_target)
        scene.step()


def set_gripper_open(q, cfg):
    q = np.array(q, dtype=float, copy=True)
    q[-2:] = cfg.gripper_open
    return q


def set_gripper_closed(q, cfg):
    q = np.array(q, dtype=float, copy=True)
    q[-2:] = cfg.gripper_closed
    return q


def open_gripper(scene, robot, cfg, hold_steps=80):
    q = set_gripper_open(to_numpy(robot.get_qpos()), cfg)
    hold_current_target(scene, robot, q, hold_steps)
    return q


def close_gripper(scene, robot, cfg, hold_steps=80):
    q = set_gripper_closed(to_numpy(robot.get_qpos()), cfg)
    hold_current_target(scene, robot, q, hold_steps)
    return q


def execute_joint_trajectory(scene, robot, q_traj, hold_steps=0):
    q_last = None
    for q in q_traj:
        q_last = np.asarray(q, dtype=float)
        robot.control_dofs_position(q_last)
        scene.step()
    if q_last is not None and hold_steps > 0:
        hold_current_target(scene, robot, q_last, hold_steps)
    return q_last


## 4. 구현 포인트 1: target pose를 joint target으로 바꾸기

이번 과제에서 첫 번째로 필요한 것은 `pose -> joint target` 변환입니다.

여기서는 직접 IK를 유도하지 않습니다.
Genesis에 들어 있는 `robot.inverse_kinematics(...)`를 사용하면 됩니다.

이 함수에서 할 일은 단순합니다.

1. `target_pos`, `target_quat`를 IK에 넣는다.
2. 나온 `q_goal`에 gripper 폭을 반영한다.
3. 최종 joint target을 반환한다.

즉, 이번 수업에서 학생이 직접 신경 쓸 부분은 `어떤 target pose를 만들 것인가`와 `그 pose들을 어떤 순서로 방문할 것인가`입니다.


In [ ]:
def solve_pose_ik(robot, ee_link, target_pos, target_quat, gripper_width=0.04):
    q_goal = to_numpy(
        robot.inverse_kinematics(
            link=ee_link,
            pos=np.asarray(target_pos, dtype=float),
            quat=np.asarray(target_quat, dtype=float),
        )
    )
    q_goal[-2:] = gripper_width
    return q_goal


## 5. 구현 포인트 2: smooth joint trajectory 만들기

이번 과제의 요구사항은 target pose를 세 개 방문하는 것만이 아니라,
그 사이를 `부드럽게` 움직이는 것입니다.

가장 쉬운 방법은 joint-space에서 cubic time scaling을 쓰는 것입니다.

여기서는 time parameter `s`를

$$
s(	au) = 3	au^2 - 2	au^3, \quad 	au \in [0,1]
$$

로 두고,

$$
q(	au) = (1-s) q_{start} + s q_{goal}
$$

로 보간합니다.

이 방식의 장점은 시작과 끝에서 속도가 0이어서,
직선 보간보다 훨씬 덜 급하게 움직인다는 점입니다.


In [ ]:
def interpolate_joint_trajectory(q_start, q_goal, num_steps):
    q_start = np.asarray(q_start, dtype=float)
    q_goal = np.asarray(q_goal, dtype=float)
    tau = np.linspace(0.0, 1.0, num_steps)
    s = 3.0 * tau**2 - 2.0 * tau**3
    q_traj = (1.0 - s[:, None]) * q_start[None, :] + s[:, None] * q_goal[None, :]
    return q_traj


## 6. 구현 포인트 3: pose sequence planner 만들기

이제 필요한 것은 여러 target pose를 순서대로 처리하는 함수입니다.

이번 과제의 동작은 아래처럼 아주 단순합니다.

- pose 1로 이동
- 1초 정지
- pose 2로 이동
- 1초 정지
- pose 3로 이동
- 1초 정지

구현 순서는 다음과 같습니다.

1. 현재 q를 시작점으로 둔다.
2. 각 pose에 대해 IK로 `q_goal`을 만든다.
3. `q_start -> q_goal` cubic trajectory를 만든다.
4. trajectory를 실행하고, 마지막 waypoint에서 1초 정지한다.
5. 다음 구간의 시작점을 방금 도착한 `q_goal`로 갱신한다.


In [ ]:
def plan_pose_sequence(scene, robot, ee_link, pose_list, cfg, gripper_width=None):
    if gripper_width is None:
        gripper_width = cfg.gripper_open

    q_current = to_numpy(robot.get_qpos())
    hold_steps = int(cfg.hold_time_sec / cfg.dt)
    q_goals = []

    for idx, pose in enumerate(pose_list):
        q_goal = solve_pose_ik(robot, ee_link, pose["pos"], pose["quat"], gripper_width=gripper_width)
        q_traj = interpolate_joint_trajectory(q_current, q_goal, cfg.steps_per_segment)
        execute_joint_trajectory(scene, robot, q_traj, hold_steps=hold_steps)
        q_current = q_goal
        q_goals.append(q_goal)
        print(f"Reached pose {idx}: pos={pose['pos']}")

    return q_goals


## 7. 4회차 실습 과제: 3개 pose를 순서대로 방문하기

아래 target pose 세 개는 marker 위치와 맞춰 두었습니다.

이번 과제에서는 아래 조건만 만족하면 됩니다.

- end-effector가 세 target pose를 순서대로 방문할 것
- 각 pose에서 1초 동안 정지할 것
- 구간 사이 움직임이 너무 급하지 않도록 smooth trajectory를 사용할 것

orientation은 모두 아래쪽을 보게 고정했습니다.
즉, 이번 시간의 핵심은 pose generation보다 `trajectory sequence`입니다.


In [ ]:
downward_quat = np.array([0.0, 1.0, 0.0, 0.0])
pose_sequence = [
    {"name": "pose_0", "pos": np.array(cfg.pose_markers[0][0], dtype=float), "quat": downward_quat},
    {"name": "pose_1", "pos": np.array(cfg.pose_markers[1][0], dtype=float), "quat": downward_quat},
    {"name": "pose_2", "pos": np.array(cfg.pose_markers[2][0], dtype=float), "quat": downward_quat},
]

open_gripper(scene, robot, cfg, hold_steps=60)
q_goals = plan_pose_sequence(scene, robot, ee_link, pose_sequence, cfg, gripper_width=cfg.gripper_open)
print("Finished all target poses.")


## 8. 여기서 final project로 어떻게 이어질까?

지금 만든 코드는 final project의 가장 작은 뼈대입니다.

여기서 바로 확장되는 요소는 아래와 같습니다.

- target pose 하나 -> pre-grasp / grasp / lift / place / retreat 여러 pose
- 단순 pose sequence -> obstacle을 피하는 via point sequence
- pose marker 세 개 -> object pose와 goal pose를 기반으로 자동 생성된 pose
- 단순 smoothness -> 수행 시간, 정지 시간, waypoint 개수 튜닝

즉, final project에서 학생들이 설계하게 될 대부분의 요소는 결국
`어떤 pose를 만들고, 어떤 순서로, 얼마나 부드럽게 연결할 것인가`로 돌아옵니다.


## 과제 아이디어

1. `pose_markers` 위치를 직접 바꿔서 더 큰 workspace를 방문하게 만들어 보세요.
2. `steps_per_segment`를 줄이거나 늘려서 motion이 어떻게 달라지는지 비교해 보세요.
3. pose 2로 바로 가지 말고, 위쪽 via point를 하나 추가해서 obstacle을 더 크게 우회하게 만들어 보세요.
4. 다음 시간 과제로는 `pre-grasp -> grasp -> lift -> place` pose sequence를 직접 만들 수 있습니다.
